# Lab 6 – Language Identification with FastText Wiki Vectors

**Dataset**: IndicCorpV2 – 23 languages, ~1000 sentences each (from Assignment 1)  
**Word Vectors**: FastText Wiki pretrained `.vec` files (300-dim, downloaded on demand)  
**Classifiers**:
1. Logistic Regression (averaged word vectors)
2. MLP with 0, 1, 2 hidden layers
3. LSTM
4. GRU
5. FastText classification model

**Evaluation**: Custom Macro-F1 (no sklearn)

## 1. Installs

!pip install fasttext-wheel numpy torch --quiet

## 2. Imports

In [1]:
import os, re, math, random, pickle, urllib.request, zipfile, tempfile
import numpy as np
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset

import fasttext

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


## 3. Load Dataset from Assignment 1 Cache

In [2]:
CACHE_FILE = os.path.join(os.getcwd(), 'dataset_sentences.pkl')
assert os.path.exists(CACHE_FILE), f'Cache not found: {CACHE_FILE}. Run lab1.ipynb first.'

with open(CACHE_FILE, 'rb') as f:
    dataset_sentences = pickle.load(f)

print(f'Loaded {len(dataset_sentences)} languages.')
for lang, sents in dataset_sentences.items():
    print(f'  {lang}: {len(sents)} sentences')

Loaded 23 languages.
  asm_Beng: 981 sentences
  ben_Beng: 981 sentences
  brx_Deva: 981 sentences
  doi_Deva: 981 sentences
  gom_Deva: 981 sentences
  guj_Gujr: 981 sentences
  hin_Deva: 981 sentences
  kan_Knda: 981 sentences
  kas_Arab: 981 sentences
  mai_Deva: 981 sentences
  mal_Mlym: 981 sentences
  mar_Deva: 981 sentences
  mni_Mtei: 981 sentences
  npi_Deva: 981 sentences
  ory_Orya: 981 sentences
  pan_Guru: 981 sentences
  san_Deva: 981 sentences
  snd_Deva: 981 sentences
  tam_Taml: 981 sentences
  tel_Telu: 981 sentences
  urd_Arab: 981 sentences
  khasi: 981 sentences
  santhali: 981 sentences


## 4. Stratified Split (80/10/10)

In [3]:
def stratified_split(dataset_sentences, train_r=0.8, val_r=0.1, test_r=0.1, seed=42):
    rng = random.Random(seed)
    train_texts, train_labels = [], []
    val_texts,   val_labels   = [], []
    test_texts,  test_labels  = [], []
    for lang, sents in dataset_sentences.items():
        s = sents[:]
        rng.shuffle(s)
        n = len(s)
        nt = int(n * train_r)
        nv = int(n * val_r)
        train_texts.extend(s[:nt]);        train_labels.extend([lang]*nt)
        val_texts.extend(s[nt:nt+nv]);     val_labels.extend([lang]*nv)
        test_texts.extend(s[nt+nv:]);      test_labels.extend([lang]*(n-nt-nv))
    return train_texts, train_labels, val_texts, val_labels, test_texts, test_labels

train_texts, train_labels, val_texts, val_labels, test_texts, test_labels = \
    stratified_split(dataset_sentences, seed=SEED)

unique_labels = sorted(set(train_labels))
label2idx = {lbl: i for i, lbl in enumerate(unique_labels)}
idx2label = {i: lbl for lbl, i in label2idx.items()}
NUM_CLASSES = len(unique_labels)

y_train = np.array([label2idx[l] for l in train_labels])
y_val   = np.array([label2idx[l] for l in val_labels])
y_test  = np.array([label2idx[l] for l in test_labels])

print(f'Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}')
print(f'Classes: {NUM_CLASSES}')

Train: 18032 | Val: 2254 | Test: 2277
Classes: 23


## 5. FastText Wiki Vector Download & Loading

We download the `.vec` text files from `https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/`  
and cache them locally. Languages without a wiki vector (Bodo, Khasi, Santhali, Meitei) fall back to a **zero vector**.

In [4]:
# Mapping: dataset language code → FastText wiki language code
# Languages with no wiki vector: brx_Deva (Bodo), mni_Mtei (Meitei), khasi, santhali
LANG_TO_FT = {
    'asm_Beng': 'as',
    'ben_Beng': 'bn',
    'brx_Deva': None,   # Bodo – no wiki vector
    'doi_Deva': 'dz',   # closest available (Dzongkha is Devanagari-adjacent); use hi as fallback
    'gom_Deva': 'gom',
    'guj_Gujr': 'gu',
    'hin_Deva': 'hi',
    'kan_Knda': 'kn',
    'kas_Arab': 'ks',
    'mai_Deva': 'mai',
    'mal_Mlym': 'ml',
    'mar_Deva': 'mr',
    'mni_Mtei': None,   # Meitei – no wiki vector
    'npi_Deva': 'ne',
    'ory_Orya': 'or',
    'pan_Guru': 'pa',
    'san_Deva': 'sa',
    'snd_Deva': 'sd',
    'tam_Taml': 'ta',
    'tel_Telu': 'te',
    'urd_Arab': 'ur',
    'khasi':    None,   # Khasi – no wiki vector
    'santhali': None,   # Santhali – no wiki vector
}

FT_BASE_URL = 'https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.{}.vec'
VEC_DIM = 300
VEC_CACHE_DIR = os.path.join(os.getcwd(), 'ft_vecs')
os.makedirs(VEC_CACHE_DIR, exist_ok=True)

def load_vec_file(lang_code, max_words=200000):
    """Load FastText .vec file for a language. Returns dict word→np.array."""
    cache_pkl = os.path.join(VEC_CACHE_DIR, f'{lang_code}.pkl')
    if os.path.exists(cache_pkl):
        with open(cache_pkl, 'rb') as f:
            return pickle.load(f)

    url = FT_BASE_URL.format(lang_code)
    vec_path = os.path.join(VEC_CACHE_DIR, f'wiki.{lang_code}.vec')

    if not os.path.exists(vec_path):
        print(f'  Downloading {url} ...', end=' ', flush=True)
        try:
            urllib.request.urlretrieve(url, vec_path)
            print('done.')
        except Exception as e:
            print(f'FAILED: {e}')
            return {}

    print(f'  Parsing {lang_code} vectors...', end=' ', flush=True)
    vecs = {}
    with open(vec_path, 'r', encoding='utf-8', errors='ignore') as f:
        first = True
        for line in f:
            if first:
                first = False
                continue  # skip header
            parts = line.rstrip().split(' ')
            if len(parts) < VEC_DIM + 1:
                continue
            word = parts[0]
            try:
                vec = np.array(parts[1:VEC_DIM+1], dtype=np.float32)
                vecs[word] = vec
            except ValueError:
                pass
            if len(vecs) >= max_words:
                break
    print(f'{len(vecs)} words.')

    with open(cache_pkl, 'wb') as f:
        pickle.dump(vecs, f)
    return vecs

print('Loading FastText wiki vectors (this may take a while on first run)...')
lang_vecs = {}  # lang_code → {word: vec}
loaded_ft_codes = set()
for lang, ft_code in LANG_TO_FT.items():
    if ft_code is None:
        lang_vecs[lang] = {}
        print(f'  {lang}: no wiki vector (zero fallback)')
    elif ft_code in loaded_ft_codes:
        # reuse already-loaded dict
        for l2, fc2 in LANG_TO_FT.items():
            if fc2 == ft_code and l2 in lang_vecs:
                lang_vecs[lang] = lang_vecs[l2]
                break
    else:
        print(f'  {lang} ({ft_code}):')
        lang_vecs[lang] = load_vec_file(ft_code)
        loaded_ft_codes.add(ft_code)

print('\nAll vectors loaded.')

Loading FastText wiki vectors (this may take a while on first run)...
  asm_Beng (as):
  ben_Beng (bn):
  brx_Deva: no wiki vector (zero fallback)
  doi_Deva (dz):
  gom_Deva (gom):
  guj_Gujr (gu):
  hin_Deva (hi):
  kan_Knda (kn):
  kas_Arab (ks):
  mai_Deva (mai):
  mal_Mlym (ml):
  mar_Deva (mr):
  mni_Mtei: no wiki vector (zero fallback)
  npi_Deva (ne):
  ory_Orya (or):
  pan_Guru (pa):
  san_Deva (sa):
  snd_Deva (sd):
  tam_Taml (ta):
  tel_Telu (te):
  urd_Arab (ur):
  khasi: no wiki vector (zero fallback)
  santhali: no wiki vector (zero fallback)

All vectors loaded.


## 6. Sentence → Averaged Word Vector

In [5]:
def sentence_to_avg_vec(text, vecs):
    """Tokenise and average word vectors. Returns zero vector if no words found."""
    words = text.split()
    vectors = [vecs[w] for w in words if w in vecs]
    if vectors:
        return np.mean(vectors, axis=0).astype(np.float32)
    return np.zeros(VEC_DIM, dtype=np.float32)

def texts_to_avg_matrix(texts, labels, lang_vecs, label2idx):
    """Build X (N, 300) and y (N,) arrays."""
    X = np.zeros((len(texts), VEC_DIM), dtype=np.float32)
    y = np.array([label2idx[l] for l in labels], dtype=np.int64)
    for i, (text, label) in enumerate(zip(texts, labels)):
        vecs = lang_vecs[label]
        X[i] = sentence_to_avg_vec(text, vecs)
    return X, y

print('Building averaged vector matrices...')
X_train, y_train = texts_to_avg_matrix(train_texts, train_labels, lang_vecs, label2idx)
X_val,   y_val   = texts_to_avg_matrix(val_texts,   val_labels,   lang_vecs, label2idx)
X_test,  y_test  = texts_to_avg_matrix(test_texts,  test_labels,  lang_vecs, label2idx)

print(f'X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')

Building averaged vector matrices...
X_train: (18032, 300), X_val: (2254, 300), X_test: (2277, 300)


## 7. Custom Macro-F1 (no sklearn)

In [6]:
def macro_f1(y_true, y_pred, num_classes):
    """Compute macro-averaged F1 score without sklearn."""
    f1s = []
    for c in range(num_classes):
        tp = np.sum((y_true == c) & (y_pred == c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        f1s.append(f1)
    return float(np.mean(f1s))

print('macro_f1 function defined.')

macro_f1 function defined.


## 8. Classifier 1 – Logistic Regression (averaged word vectors)

Implemented from scratch using softmax + cross-entropy + gradient descent.

In [7]:
class LogisticRegressionScratch:
    """Multinomial logistic regression (softmax) implemented from scratch."""

    def __init__(self, input_dim, num_classes, lr=0.1, epochs=100, batch_size=256, seed=42):
        rng = np.random.RandomState(seed)
        scale = np.sqrt(2.0 / input_dim)
        self.W = rng.randn(input_dim, num_classes).astype(np.float32) * scale
        self.b = np.zeros(num_classes, dtype=np.float32)
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size

    def _softmax(self, z):
        z = z - z.max(axis=1, keepdims=True)  # numerical stability
        exp_z = np.exp(z)
        return exp_z / exp_z.sum(axis=1, keepdims=True)

    def fit(self, X, y, X_val=None, y_val=None):
        N = len(X)
        best_val_f1, best_W, best_b = -1, self.W.copy(), self.b.copy()
        for epoch in range(self.epochs):
            # Shuffle
            idx = np.random.permutation(N)
            X, y = X[idx], y[idx]
            for start in range(0, N, self.batch_size):
                Xb = X[start:start+self.batch_size]
                yb = y[start:start+self.batch_size]
                # Forward
                logits = Xb @ self.W + self.b
                probs  = self._softmax(logits)
                # One-hot
                one_hot = np.zeros_like(probs)
                one_hot[np.arange(len(yb)), yb] = 1.0
                # Gradients
                diff = (probs - one_hot) / len(yb)
                dW = Xb.T @ diff
                db = diff.sum(axis=0)
                self.W -= self.lr * dW
                self.b -= self.lr * db
            if X_val is not None and (epoch + 1) % 10 == 0:
                preds = self.predict(X_val)
                f1 = macro_f1(y_val, preds, num_classes=self.W.shape[1])
                if f1 > best_val_f1:
                    best_val_f1 = f1
                    best_W, best_b = self.W.copy(), self.b.copy()
                print(f'  Epoch {epoch+1:3d}: val macro-F1 = {f1:.4f}')
        if X_val is not None:
            self.W, self.b = best_W, best_b

    def predict(self, X):
        logits = X @ self.W + self.b
        return np.argmax(logits, axis=1)


print('Training Logistic Regression...')
lr_model = LogisticRegressionScratch(
    input_dim=VEC_DIM, num_classes=NUM_CLASSES, lr=0.5, epochs=100, batch_size=256
)
lr_model.fit(X_train.copy(), y_train.copy(), X_val, y_val)

lr_preds = lr_model.predict(X_test)
lr_f1 = macro_f1(y_test, lr_preds, NUM_CLASSES)
print(f'\nLogistic Regression – Test Macro-F1: {lr_f1:.4f}')

Training Logistic Regression...
  Epoch  10: val macro-F1 = 0.7749
  Epoch  20: val macro-F1 = 0.7749
  Epoch  30: val macro-F1 = 0.7749
  Epoch  40: val macro-F1 = 0.7749
  Epoch  50: val macro-F1 = 0.7749
  Epoch  60: val macro-F1 = 0.7749
  Epoch  70: val macro-F1 = 0.7749
  Epoch  80: val macro-F1 = 0.7749
  Epoch  90: val macro-F1 = 0.7749
  Epoch 100: val macro-F1 = 0.7749

Logistic Regression – Test Macro-F1: 0.7770


## 9. Classifier 2 – Multi-Layer Perceptron (n=0, 1, 2 hidden layers)

Built with PyTorch. Averaged word vectors as input features.

In [8]:
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_sizes=None):
        super().__init__()
        hidden_sizes = hidden_sizes or []
        layers = []
        prev = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.3)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_mlp(hidden_sizes, X_tr, y_tr, X_vl, y_vl, X_te, y_te,
              lr=1e-3, epochs=50, batch_size=256, patience=10):
    model = MLP(VEC_DIM, NUM_CLASSES, hidden_sizes).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    Xtr_t = torch.tensor(X_tr).to(DEVICE)
    ytr_t = torch.tensor(y_tr).to(DEVICE)
    Xvl_t = torch.tensor(X_vl).to(DEVICE)
    yvl_t = torch.tensor(y_vl)
    Xte_t = torch.tensor(X_te).to(DEVICE)

    ds = TensorDataset(Xtr_t, ytr_t)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)

    best_f1, best_state, no_improve = -1, None, 0
    for epoch in range(epochs):
        model.train()
        for Xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds = model(Xvl_t).argmax(dim=1).cpu().numpy()
        f1 = macro_f1(y_vl, val_preds, NUM_CLASSES)
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            print(f'  Early stop at epoch {epoch+1}')
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_preds = model(Xte_t).argmax(dim=1).cpu().numpy()
    test_f1 = macro_f1(y_te, test_preds, NUM_CLASSES)
    return test_f1


mlp_configs = {
    'MLP (n=0, linear)': [],
    'MLP (n=1, 512)':    [512],
    'MLP (n=2, 512-256)':[512, 256],
}

mlp_results = {}
for name, hidden in mlp_configs.items():
    print(f'Training {name}...')
    f1 = train_mlp(hidden, X_train, y_train, X_val, y_val, X_test, y_test,
                   lr=1e-3, epochs=50, batch_size=256, patience=10)
    mlp_results[name] = f1
    print(f'  Test Macro-F1: {f1:.4f}')

print('\nMLP Results:')
for name, f1 in mlp_results.items():
    print(f'  {name}: {f1:.4f}')

Training MLP (n=0, linear)...
  Early stop at epoch 13
  Test Macro-F1: 0.7770
Training MLP (n=1, 512)...
  Early stop at epoch 11
  Test Macro-F1: 0.7770
Training MLP (n=2, 512-256)...
  Early stop at epoch 12
  Test Macro-F1: 0.7770

MLP Results:
  MLP (n=0, linear): 0.7770
  MLP (n=1, 512): 0.7770
  MLP (n=2, 512-256): 0.7770


## 10. Sequence Dataset (word vectors per sentence)

For LSTM/GRU we feed each sentence as a sequence of word vectors.

In [9]:
MAX_SEQ_LEN = 50

def texts_to_sequences(texts, labels, lang_vecs, label2idx, max_len=MAX_SEQ_LEN):
    """Convert texts to padded (N, max_len, 300) tensor and labels."""
    X = np.zeros((len(texts), max_len, VEC_DIM), dtype=np.float32)
    y = np.array([label2idx[l] for l in labels], dtype=np.int64)
    for i, (text, label) in enumerate(zip(texts, labels)):
        vecs = lang_vecs[label]
        words = text.split()
        seq = [vecs[w] for w in words if w in vecs]
        seq = seq[:max_len]
        for j, v in enumerate(seq):
            X[i, j] = v
    return X, y

print('Building sequence tensors...')
Xs_train, ys_train = texts_to_sequences(train_texts, train_labels, lang_vecs, label2idx)
Xs_val,   ys_val   = texts_to_sequences(val_texts,   val_labels,   lang_vecs, label2idx)
Xs_test,  ys_test  = texts_to_sequences(test_texts,  test_labels,  lang_vecs, label2idx)

print(f'Xs_train: {Xs_train.shape}')

Building sequence tensors...
Xs_train: (18032, 50, 300)


## 11. Classifier 3 – LSTM

In [10]:
class RNNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, rnn_type='LSTM',
                 num_layers=1, dropout=0.3):
        super().__init__()
        rnn_cls = nn.LSTM if rnn_type == 'LSTM' else nn.GRU
        self.rnn = rnn_cls(
            input_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0.0
        )
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_dim, num_classes)
        self.rnn_type = rnn_type

    def forward(self, x):
        out, hidden = self.rnn(x)
        if self.rnn_type == 'LSTM':
            last_h = hidden[0][-1]  # (batch, hidden)
        else:
            last_h = hidden[-1]
        return self.fc(self.drop(last_h))


def train_rnn(rnn_type, Xs_tr, ys_tr, Xs_vl, yvl, Xs_te, yte,
              hidden_dim=256, lr=1e-3, epochs=30, batch_size=128, patience=5):
    model = RNNClassifier(VEC_DIM, hidden_dim, NUM_CLASSES, rnn_type=rnn_type).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    Xtr_t = torch.tensor(Xs_tr).to(DEVICE)
    ytr_t = torch.tensor(ys_tr).to(DEVICE)
    Xvl_t = torch.tensor(Xs_vl).to(DEVICE)
    Xte_t = torch.tensor(Xs_te).to(DEVICE)

    ds = TensorDataset(Xtr_t, ytr_t)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)

    best_f1, best_state, no_improve = -1, None, 0
    for epoch in range(epochs):
        model.train()
        for Xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds = model(Xvl_t).argmax(dim=1).cpu().numpy()
        f1 = macro_f1(yvl, val_preds, NUM_CLASSES)
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            print(f'  Early stop at epoch {epoch+1}')
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_preds = model(Xte_t).argmax(dim=1).cpu().numpy()
    return macro_f1(yte, test_preds, NUM_CLASSES)


print('Training LSTM...')
lstm_f1 = train_rnn('LSTM', Xs_train, ys_train, Xs_val, ys_val, Xs_test, ys_test,
                    hidden_dim=256, lr=1e-3, epochs=30, batch_size=128, patience=5)
print(f'LSTM – Test Macro-F1: {lstm_f1:.4f}')

Training LSTM...
  Early stop at epoch 23
LSTM – Test Macro-F1: 0.5169


## 12. Classifier 4 – GRU

In [11]:
print('Training GRU...')
gru_f1 = train_rnn('GRU', Xs_train, ys_train, Xs_val, ys_val, Xs_test, ys_test,
                   hidden_dim=256, lr=1e-3, epochs=30, batch_size=128, patience=5)
print(f'GRU – Test Macro-F1: {gru_f1:.4f}')

Training GRU...
  Early stop at epoch 12
GRU – Test Macro-F1: 0.7754


## 13. Classifier 5 – FastText Classification

Uses the `fasttext` library's supervised training directly on text.  
Macro-F1 computed from its predictions using our custom function.

In [13]:
FT_TRAIN_FILE = os.path.join(os.getcwd(), 'ft_train.txt')
FT_VAL_FILE   = os.path.join(os.getcwd(), 'ft_val.txt')
FT_TEST_FILE  = os.path.join(os.getcwd(), 'ft_test.txt')

def write_fasttext_file(texts, labels, path):
    """Write FastText supervised format: __label__<lang> <text>."""
    with open(path, 'w', encoding='utf-8') as f:
        for text, label in zip(texts, labels):
            # Sanitize: remove newlines
            clean = text.replace('\n', ' ').replace('\r', ' ')
            f.write(f'__label__{label} {clean}\n')

write_fasttext_file(train_texts, train_labels, FT_TRAIN_FILE)
write_fasttext_file(val_texts,   val_labels,   FT_VAL_FILE)
write_fasttext_file(test_texts,  test_labels,  FT_TEST_FILE)
print('FastText data files written.')

# Train FastText model
print('Training FastText classifier...')
ft_model = fasttext.train_supervised(
    input=FT_TRAIN_FILE,
    epoch=25,
    lr=0.5,
    wordNgrams=2,
    dim=100,
    minCount=1,
    loss='softmax',
    thread=4,
    verbose=0
)
print('FastText training done.')

# Predict on test set using batch prediction (avoids numpy 2.x copy=False issue)
clean_texts = [t.replace('\n', ' ').replace('\r', ' ') for t in test_texts]
labels_batch, _ = ft_model.predict(clean_texts, k=1)
ft_pred_labels = [lbl[0].replace('__label__', '') for lbl in labels_batch]
ft_pred_idx    = np.array([label2idx.get(l, 0) for l in ft_pred_labels])
ft_f1 = macro_f1(y_test, ft_pred_idx, NUM_CLASSES)
print(f'FastText Classification – Test Macro-F1: {ft_f1:.4f}')


FastText data files written.
Training FastText classifier...
FastText training done.
FastText Classification – Test Macro-F1: 0.8584


## 14. Results Summary

In [14]:
results = {
    'Logistic Regression (avg word vec)': lr_f1,
}
results.update(mlp_results)
results['LSTM'] = lstm_f1
results['GRU']  = gru_f1
results['FastText Classification'] = ft_f1

print('=' * 55)
print(f'  {"Classifier":<38} {"Macro-F1":>8}')
print('=' * 55)
for name, score in results.items():
    print(f'  {name:<38} {score:>8.4f}')
print('=' * 55)

best = max(results, key=results.get)
print(f'\nBest classifier: {best} (Macro-F1 = {results[best]:.4f})')

  Classifier                             Macro-F1
  Logistic Regression (avg word vec)       0.7770
  MLP (n=0, linear)                        0.7770
  MLP (n=1, 512)                           0.7770
  MLP (n=2, 512-256)                       0.7770
  LSTM                                     0.5169
  GRU                                      0.7754
  FastText Classification                  0.8584

Best classifier: FastText Classification (Macro-F1 = 0.8584)
